# Redes e árvores com protocolo comum e cinco sementes
Complemento 03A.05C, **experimento executado no Colab: 20 ajustes concluídos e conferidos**.
Executar no Colab depois de 03A.05B, na ordem. Não depende da chave EPA.
Leva mais tempo que a análise de previsões. Execute uma vez; preserve o ZIP final.

Comparação condicional às configurações já definidas no projeto, em amostra
comum com histórico ampliado: A (uma entrada) e B (sete entradas), MLP e XGBoost,
sementes **42, 43, 44, 45 e 46**. Uma configuração por família e cenário, cinco
repetições: 20 ajustes. Não é busca de hiperparâmetros nem reprodução dos
experimentos anteriores, que utilizaram outros dados ou reajuste com outubro.

Mesmas 88.475 linhas de treino, 211 de validação em outubro, 1.144 de teste.
Para **ambas** as famílias, escolher a complexidade em outubro e aplicar o modelo
diretamente ao teste, sem reajuste incluindo a validação. Nenhuma semente é
escolhida pelo teste. Mesma quantidade de configurações e repetições não significa
mesmo tempo computacional; registrar tempo, parâmetros/árvores e épocas.
O teste já foi reutilizado, e cinco sementes não constituem validação externa.

In [1]:
# Executar antes dos imports do XGBoost; TensorFlow vem instalado no Colab.
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','xgboost==3.4.1','pyarrow'])

0

In [2]:
import os, sys, json, hashlib, zipfile, io, platform, tempfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

# Local: PROJECT_ROOT; Colab: pasta do projeto no MyDrive.
# Alternativa portátil: enviar 03A05BC_insumos.zip quando solicitado.
if 'google.colab' in sys.modules or os.path.isdir('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
default_root = Path('/content/drive/MyDrive/co2-cems-thermal-modeling')
PORTABLE_INPUTS = False
PROJECT = Path(os.environ.get('PROJECT_ROOT', str(default_root)))
if not (PROJECT/'data/interim/03A05_trajectory_history').exists():
    nested = PROJECT/'co2-cems-thermal-modeling'
    if (nested/'data/interim/03A05_trajectory_history').exists():
        PROJECT = nested
if not (PROJECT/'data/interim/03A05_trajectory_history').exists():
    from google.colab import files
    print('Envie o arquivo 03A05BC_insumos.zip que acompanha os notebooks.')
    uploaded = files.upload()
    archives = [k for k in uploaded if k.endswith('.zip')]
    assert len(archives) == 1, 'Envie apenas o ZIP de insumos.'
    PROJECT = Path(tempfile.mkdtemp(prefix='co2_insumos_'))
    PORTABLE_INPUTS = True
    with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as z:
        for item in z.infolist():
            dest = (PROJECT/item.filename).resolve()
            assert dest.is_relative_to(PROJECT.resolve()), 'Caminho inseguro no ZIP.'
        z.extractall(PROJECT)
EXPECTED_HASHES = {'data/interim/03A05_trajectory_history/03A05_treino_selecao_A_B.parquet': '7d7aebca5a1d588fe6bff48e25002693724644d0ef5e8836bb408dd1a021080a', 'data/interim/03A05_trajectory_history/03A05_validacao_outubro_2025_A_B.parquet': 'aa54c3b08d43f1151244dbd4d83ebf2673a52a6019c6f8f100190aec6552a54b', 'data/interim/03A05_trajectory_history/03A05_teste_nov_dez_2025_A_B.parquet': '526c80faafd6bd9f22a53197acb90bf06fd8800cbba0443a05c21bf9b9a32c70', 'data/interim/03A05_trajectory_history/03A05_previsoes_pareadas_com_contextos.parquet': 'eb204d81dae4ccc3f073fbe630bd88283bb8a9c3f774dca5bae5b36ac2c699be', 'outputs/03A05A_ablacao_controlada.zip': 'd1f91f7717a7302947fe74e4db839ac4db87219e39281c7ab8e0ce8e7bb2ffa1'}
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def validate_inputs(paths):
    actual = {p: sha256(PROJECT/p) for p in paths}
    for p,h in actual.items():
        assert h == EXPECTED_HASHES[p], f'Insumo diferente da execução recebida: {p}'
    return actual
BASE = PROJECT/'data/interim/03A05_trajectory_history'
TARGET = 'CO2 Mass (short tons)'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')

Mounted at /content/drive


In [3]:
INPUT_PATHS = [
    'data/interim/03A05_trajectory_history/03A05_treino_selecao_A_B.parquet',
    'data/interim/03A05_trajectory_history/03A05_validacao_outubro_2025_A_B.parquet',
    'data/interim/03A05_trajectory_history/03A05_teste_nov_dez_2025_A_B.parquet']
INPUT_HASHES=validate_inputs(INPUT_PATHS)
train, val, test = [pd.read_parquet(PROJECT/p).sort_values('timestamp').reset_index(drop=True) for p in INPUT_PATHS]
FEATURES={'A':['load_current_mw'],'B':['load_current_mw','load_lag_1h','delta_load_1h',
    'abs_delta_load_1h','load_mean_previous_3h','operating_time_lag_1h','hours_since_startup']}
SEEDS=[42,43,44,45,46]
assert [len(d) for d in [train,val,test]]==[88475,211,1144]
assert train.timestamp.max()<val.timestamp.min()<test.timestamp.min()
assert val.timestamp.max()<test.timestamp.min()
for d in [train,val,test]:
    assert d.timestamp.is_unique
    assert np.isfinite(d[FEATURES['B']+[TARGET]].to_numpy()).all()
assert val.timestamp.min()>=pd.Timestamp('2025-10-01')
assert val.timestamp.max()<pd.Timestamp('2025-11-01')
assert test.timestamp.min()==pd.Timestamp('2025-11-01')
OUTPUT=PROJECT/'outputs'/'03A05C_sementes_protocolo_comum'/RUN_ID
OUTPUT.mkdir(parents=True,exist_ok=False)
print('Validação dos insumos concluída. Próxima célula define o protocolo; a seguinte treina 20 modelos.')

Validação dos insumos concluída. Próxima célula define o protocolo; a seguinte treina 20 modelos.


## Configurações e controle de execução
MLP 64/32 ReLU e saída linear; Adam 0,001; MSE; batch 64; até 500 épocas;
paciência 30; min_delta 10⁻⁶; shuffle=False; recuperação dos melhores pesos.
MinMaxScaler em X e y ajustado só no treino. Val_loss escalada corresponde
à mesma ordenação do RMSE original porque o scaler de y é afim e comum.
XGBoost com configuração de referência, até 3.000 árvores, paciência 50 e RMSE.
Limites por família preservam as configurações do projeto, não igualam operações.

CPU para as duas famílias; quatro threads, quando disponíveis. Determinismo do
TensorFlow é solicitado e o ambiente é registrado. Mesma semente não garante
previsões idênticas entre plataformas/versões. Tempos são desta sessão, sem
warm-up padronizado; não constituem benchmark de hardware. Não extrapolar a
comparação de estabilidade a outras arquiteturas, históricos ou plantas.

In [4]:
os.environ['CUDA_VISIBLE_DEVICES']='-1'
os.environ['TF_DETERMINISTIC_OPS']='1'
import time, importlib.metadata as metadata, joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
THREADS=min(4,os.cpu_count() or 1)
assert xgb.__version__=='3.4.1', 'Reinicie a sessão após instalar XGBoost e execute desde o começo.'
tf.config.set_visible_devices([], 'GPU')
tf.config.threading.set_intra_op_parallelism_threads(THREADS)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.experimental.enable_op_determinism()
XGB_PARAMS=dict(objective='reg:squarederror',n_estimators=3000,learning_rate=.03,max_depth=4,
    min_child_weight=5,subsample=.8,colsample_bytree=.8,reg_alpha=0.,reg_lambda=1.,
    eval_metric='rmse',early_stopping_rounds=50,n_jobs=THREADS,tree_method='hist',device='cpu')
MLP_PARAMS=dict(hidden=[64,32],learning_rate=.001,batch_size=64,max_epochs=500,
    patience=30,min_delta=1e-6,shuffle=False,restore_best_weights=True)
ENV={k:metadata.version(k) for k in ['tensorflow','keras','xgboost','scikit-learn','numpy','pandas','pyarrow']}
protocol=dict(experimento='novo; protocolo comum sem reajuste com outubro',sementes=SEEDS,
    cenarios=FEATURES,n_treino=len(train),n_validacao=len(val),n_teste=len(test),
    xgb=XGB_PARAMS,mlp=MLP_PARAMS,configs_por_familia_cenario=1,ajustes_planejados=20,
    cpu_threads=THREADS,python=sys.version,platform=platform.platform(),versoes=ENV,
    hashes_insumos=INPUT_HASHES,teste_reutilizado=True,criterio='validacao MSE/RMSE; sem selecao por teste')
(OUTPUT/'protocolo_antes_do_treino.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(protocol,ensure_ascii=False,indent=2))

{
  "experimento": "novo; protocolo comum sem reajuste com outubro",
  "sementes": [
    42,
    43,
    44,
    45,
    46
  ],
  "cenarios": {
    "A": [
      "load_current_mw"
    ],
    "B": [
      "load_current_mw",
      "load_lag_1h",
      "delta_load_1h",
      "abs_delta_load_1h",
      "load_mean_previous_3h",
      "operating_time_lag_1h",
      "hours_since_startup"
    ]
  },
  "n_treino": 88475,
  "n_validacao": 211,
  "n_teste": 1144,
  "xgb": {
    "objective": "reg:squarederror",
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "eval_metric": "rmse",
    "early_stopping_rounds": 50,
    "n_jobs": 2,
    "tree_method": "hist",
    "device": "cpu"
  },
  "mlp": {
    "hidden": [
      64,
      32
    ],
    "learning_rate": 0.001,
    "batch_size": 64,
    "max_epochs": 500,
    "patience": 30,
    "min_delta": 1e-06,


In [5]:
# Executar sequencialmente. Os resultados parciais são gravados após cada ajuste.
# Não continuar a partir do meio após reinício: iniciar uma nova execução completa.
rows=[]; predictions=[]
def evaluate(y,p):
    y=np.asarray(y,dtype='float64'); p=np.asarray(p,dtype='float64'); e=p-y
    return dict(MAE=float(np.abs(e).mean()),RMSE=float(np.sqrt(np.mean(e**2))),
        R2=float(1-np.sum(e**2)/np.sum((y-y.mean())**2)),vies_observado_menos_pred=float(-e.mean()))
for seed in SEEDS:
    for scenario,features in FEATURES.items():
        # Alternar a ordem evita concentrar sempre o aquecimento na mesma família.
        families=['MLP','XGBoost'] if seed%2==0 else ['XGBoost','MLP']
        for family in families:
            label=f'{family}_{scenario}_seed{seed}'
            t0=time.perf_counter()
            if family=='MLP':
                tf.keras.backend.clear_session()
                tf.keras.utils.set_random_seed(seed)
                sx,sy=MinMaxScaler(),MinMaxScaler()
                xt=sx.fit_transform(train[features]).astype('float32')
                xv=sx.transform(val[features]).astype('float32')
                xe=sx.transform(test[features]).astype('float32')
                yt=sy.fit_transform(train[[TARGET]]).astype('float32')
                yv=sy.transform(val[[TARGET]]).astype('float32')
                preprocessing_s=time.perf_counter()-t0
                model=keras.Sequential([keras.layers.Input(shape=(len(features),)),
                    keras.layers.Dense(64,activation='relu'),keras.layers.Dense(32,activation='relu'),
                    keras.layers.Dense(1)])
                model.compile(optimizer=keras.optimizers.Adam(.001),loss='mse')
                early=keras.callbacks.EarlyStopping(monitor='val_loss',patience=30,min_delta=1e-6,
                    restore_best_weights=True)
                tfopt=tf.data.Options(); tfopt.threading.private_threadpool_size=THREADS
                tfopt.threading.max_intra_op_parallelism=THREADS
                train_ds=tf.data.Dataset.from_tensor_slices((xt,yt)).batch(64).with_options(tfopt)
                val_ds=tf.data.Dataset.from_tensor_slices((xv,yv)).batch(64).with_options(tfopt)
                begin=time.perf_counter()
                history=model.fit(train_ds,validation_data=val_ds,epochs=500,shuffle=False,
                    callbacks=[early],verbose=0)
                fit_s=time.perf_counter()-begin
                hist=pd.DataFrame(history.history)
                hist.to_csv(OUTPUT/(label+'_historico.csv'),index_label='epoca_base_zero')
                best=int(early.best_epoch)+1
                trained=len(hist)
                complexity=int(model.count_params())
                begin=time.perf_counter()
                pv=sy.inverse_transform(model(xv,training=False).numpy()).ravel()
                pt=sy.inverse_transform(model(xe,training=False).numpy()).ravel()
                predict_s=time.perf_counter()-begin
                model.save(OUTPUT/(label+'.keras'))
                joblib.dump({'X':sx,'y':sy,'features':features},OUTPUT/(label+'_scalers.joblib'))
            else:
                xt,xv,xe=[d[features] for d in [train,val,test]]
                yt,yv=train[TARGET],val[TARGET]
                preprocessing_s=time.perf_counter()-t0
                model=xgb.XGBRegressor(**XGB_PARAMS,random_state=seed)
                begin=time.perf_counter()
                model.fit(xt,yt,eval_set=[(xv,yv)],verbose=False)
                fit_s=time.perf_counter()-begin
                hist=pd.DataFrame(model.evals_result()['validation_0'])
                hist.to_csv(OUTPUT/(label+'_historico.csv'),index_label='iteracao_base_zero')
                best=int(model.best_iteration)+1
                trained=int(model.get_booster().num_boosted_rounds())
                complexity=best
                begin=time.perf_counter()
                # iteration_range explícito evita usar as árvores de paciência.
                pv=model.predict(xv,iteration_range=(0,best))
                pt=model.predict(xe,iteration_range=(0,best))
                predict_s=time.perf_counter()-begin
                model.save_model(OUTPUT/(label+'.json'))
            assert np.isfinite(pt).all() and np.isfinite(pv).all()
            for split,actual,pred in [('validacao',val[TARGET],pv),('teste',test[TARGET],pt)]:
                rows.append(dict(familia=family,cenario=scenario,semente=seed,particao=split,
                    n=len(actual),melhor_epoca_ou_arvores=best,epocas_ou_arvores_treinadas=trained,
                    parametros_mlp_ou_arvores_usadas=complexity,preprocessamento_s=preprocessing_s,
                    ajuste_s=fit_s,predicao_validacao_e_teste_s=predict_s,**evaluate(actual,pred)))
            predictions.append(pd.DataFrame(dict(timestamp=test.timestamp,y=test[TARGET],
                previsao=pt,familia=family,cenario=scenario,semente=seed)))
            pd.DataFrame(rows).to_csv(OUTPUT/'metricas_e_tempos_parciais.csv',index=False)
            pd.concat(predictions,ignore_index=True).to_parquet(OUTPUT/'previsoes_teste.parquet',index=False)
            print(label,'concluído | ajuste',round(fit_s,2),'s | complexidade selecionada',best,flush=True)
assert len(rows)==40 and len(predictions)==20
metrics=pd.DataFrame(rows)
metrics.to_csv(OUTPUT/'metricas_e_tempos.csv',index=False)
summary=metrics.groupby(['familia','cenario','particao']).agg(
    sementes=('semente','nunique'),MAE_media=('MAE','mean'),MAE_dp_sementes=('MAE','std'),
    MAE_min=('MAE','min'),MAE_max=('MAE','max'),RMSE_media=('RMSE','mean'),
    ajuste_s_media=('ajuste_s','mean'),ajuste_s_total=('ajuste_s','sum')).reset_index()
summary.to_csv(OUTPUT/'resumo_sementes.csv',index=False)
# Diferenças pareadas por semente: não escolher a melhor semente do teste.
paired=metrics[metrics.particao=='teste'].pivot(index=['cenario','semente'],columns='familia',values='MAE').reset_index()
paired['MAE_MLP_menos_XGBoost']=paired.MLP-paired.XGBoost
paired.to_csv(OUTPUT/'diferencas_familias_por_semente.csv',index=False)
assert validate_inputs(INPUT_PATHS)==INPUT_HASHES
(OUTPUT/'CONCLUIDO.json').write_text(json.dumps(dict(ajustes_concluidos=20,protocolo=protocol),ensure_ascii=False,indent=2),encoding='utf-8')
import shutil
archive=shutil.make_archive(str(OUTPUT.parent/('03A05C_'+RUN_ID)),'zip',root_dir=OUTPUT)
display(summary.round(6));display(paired.round(6))
print('Resultados:',OUTPUT)
print('ZIP para compartilhar:',archive)
if PORTABLE_INPUTS:
    from google.colab import files
    files.download(archive)

MLP_A_seed42 concluído | ajuste 85.22 s | complexidade selecionada 1
XGBoost_A_seed42 concluído | ajuste 5.98 s | complexidade selecionada 498
MLP_B_seed42 concluído | ajuste 163.62 s | complexidade selecionada 30
XGBoost_B_seed42 concluído | ajuste 4.85 s | complexidade selecionada 721
XGBoost_A_seed43 concluído | ajuste 1.6 s | complexidade selecionada 398
MLP_A_seed43 concluído | ajuste 141.46 s | complexidade selecionada 24
XGBoost_B_seed43 concluído | ajuste 8.37 s | complexidade selecionada 868
MLP_B_seed43 concluído | ajuste 140.29 s | complexidade selecionada 23
MLP_A_seed44 concluído | ajuste 77.96 s | complexidade selecionada 1
XGBoost_A_seed44 concluído | ajuste 1.49 s | complexidade selecionada 356
MLP_B_seed44 concluído | ajuste 135.06 s | complexidade selecionada 19
XGBoost_B_seed44 concluído | ajuste 7.75 s | complexidade selecionada 725
XGBoost_A_seed45 concluído | ajuste 1.78 s | complexidade selecionada 449
MLP_A_seed45 concluído | ajuste 124.35 s | complexidade selec

,familia,cenario,particao,sementes,MAE_media,MAE_dp_sementes,MAE_min,MAE_max,RMSE_media,ajuste_s_media,ajuste_s_total
0,MLP,A,teste,5,57.282686,1.248247,55.950528,58.919772,60.508778,111.042645,555.213225
1,MLP,A,validacao,5,34.080292,0.365926,33.677065,34.592119,44.850085,111.042645,555.213225
2,MLP,B,teste,5,56.382355,1.820140,54.194079,58.634650,60.379398,143.656038,718.280191
3,MLP,B,validacao,5,33.486383,0.523880,32.750111,34.047832,44.342574,143.656038,718.280191
4,XGBoost,A,teste,5,20.451809,0.024797,20.420230,20.480388,25.269438,2.528311,12.641554
5,XGBoost,A,validacao,5,59.922110,0.022253,59.903668,59.955080,75.504217,2.528311,12.641554
6,XGBoost,B,teste,5,21.223633,0.617455,20.423802,22.139184,26.615099,6.590000,32.950001
7,XGBoost,B,validacao,5,53.544406,0.249825,53.256287,53.797289,67.600959,6.590000,32.950001


familia,cenario,semente,MLP,XGBoost,MAE_MLP_menos_XGBoost
0,A,42,57.165794,20.463616,36.702179
1,A,43,56.254203,20.480388,35.773814
2,A,44,55.950528,20.420230,35.530297
3,A,45,58.123131,20.432120,37.691011
4,A,46,58.919772,20.462692,38.457080
5,B,42,54.194079,21.039361,33.154718
6,B,43,57.022180,21.331168,35.691012
7,B,44,57.199448,21.184652,36.014796
8,B,45,54.861420,22.139184,32.722236
9,B,46,58.634650,20.423802,38.210848


Resultados: /content/drive/MyDrive/co2-cems-thermal-modeling/outputs/03A05C_sementes_protocolo_comum/20261030T020641_212185Z
ZIP para compartilhar: /content/drive/MyDrive/co2-cems-thermal-modeling/outputs/03A05C_sementes_protocolo_comum/03A05C_20261030T020641_212185Z.zip


## Como salvar e interpretar
A última célula informa o caminho do ZIP dentro de outputs/03A05C_sementes_protocolo_comum.
No Colab, salve também o notebook executado: Arquivo → Fazer download → .ipynb.
Envie o ZIP e o notebook executado para inspeção. Modelos e históricos são salvos
individualmente; CONCLUIDO.json confirma os 20 ajustes. Uma execução interrompida
mantém arquivos parciais e não deve ser apresentada como completa.

O desvio entre sementes descreve apenas a aleatoriedade sob este protocolo;
não é intervalo temporal ou erro de generalização para outra planta. Os resultados
não devem substituir automaticamente as tabelas anteriores. Examinar todas as
sementes, validação e teste, sem selecionar retrospectivamente a melhor combinação.

## Referências
GRINSZTAJN, L.; OYALLON, E.; VAROQUAUX, G. Why do tree-based models still outperform
deep learning on typical tabular data? NeurIPS, 35, 2022. https://doi.org/10.52202/068431-0037.
Motiva documentar orçamento e dados tabulares; não demonstra superioridade na Iatan.

SI, M.; TARNOCZI, T. J.; WIENS, B. M.; DU, K. Development of Predictive Emissions
Monitoring System Using Open Source Machine Learning Library – Keras: A Case Study
on a Cogeneration Unit. IEEE Access, 7, 113463–113475, 2019.
https://doi.org/10.1109/ACCESS.2019.2930555. Aplicação de redes em PEMS de NOx;
não reproduzimos sua divisão aleatória nem transferimos seu coeficiente r para R².

RUMELHART, D. E.; HINTON, G. E.; WILLIAMS, R. J. Learning representations by
back-propagating errors. Nature, 323, 533–536, 1986. https://doi.org/10.1038/323533a0.
GLOROT, X.; BORDES, A.; BENGIO, Y. Deep Sparse Rectifier Neural Networks.
AISTATS, PMLR 15, 315–323, 2011. https://proceedings.mlr.press/v15/glorot11a.html.
KINGMA, D. P.; BA, J. Adam: A Method for Stochastic Optimization. ICLR, 2015.
https://arxiv.org/abs/1412.6980.
CHEN, T.; GUESTRIN, C. XGBoost: A Scalable Tree Boosting System.
KDD, 785–794, 2016. https://doi.org/10.1145/2939672.2939785.

Documentação das APIs (consultada em 29/10/2026):
https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping ;
https://www.tensorflow.org/api_docs/python/tf/config/experimental/enable_op_determinism ;
https://xgboost.readthedocs.io/en/stable/python/python_api.html ;
https://scikit-learn.org/1.6/modules/generated/sklearn.preprocessing.MinMaxScaler.html.
As escolhas de cinco sementes, configuração fixa, partições e CPU são próprias
deste complemento; não constituem protocolo copiado dos artigos citados.

## Interpretação da execução conferida

Os 20 ajustes terminaram. Em cada cenário e semente, o MAE de teste do XGBoost foi menor que o da MLP. Médias ± desvios padrão entre sementes: XGBoost A 20,4518 ± 0,0248; XGBoost B 21,2236 ± 0,6175; MLP A 57,2827 ± 1,2482; MLP B 56,3824 ± 1,8201 short tons por registro horário. Na validação de outubro, a MLP teve menor MAE em todos os pares: a ordenação se inverte entre períodos.

B melhora o MAE em três de cinco sementes da MLP e uma de cinco do XGBoost. A dispersão entre sementes é condicional às configurações e às mesmas 1.144 horas de teste. Não mede incerteza temporal, não fornece dez avaliações temporais independentes e não resolve a reutilização do teste. O protocolo não reajusta os modelos com outubro; seus valores não substituem o 03A.05 final.

Foram aprovadas 376 verificações dos três complementos, incluindo recálculo das métricas, identidade dos insumos e recarga dos 20 modelos. Igualdade exata de previsões para XGBoost; diferença máxima de 0,0001221 short ton para MLP entre plataformas, abaixo de 0,001. Nenhum novo treinamento foi necessário nessa conferência.
